In [ ]:
import os
import random
from google.cloud import texttospeech
from dotenv import load_dotenv

# Load .env if needed
load_dotenv()

# Set Google Credentials
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../serviceAccount.json"

# Init client
client = texttospeech.TextToSpeechClient()

# Output folder
output_dir = "../data/synthetic/samples/wav"
os.makedirs(output_dir, exist_ok=True)

# Load synthetic text located in ../data/synthetic/synthetic_medical_corpus.txt
with open("../data/synthetic/synthetic_medical_corpus.txt", "r", encoding="utf-8") as f:
    phrases = [line.strip() for line in f.readlines() if line.strip()]

# Define available French voices
voices = [
    {"name": "fr-FR-Wavenet-B", "gender": texttospeech.SsmlVoiceGender.MALE},    # Male
    {"name": "fr-FR-Wavenet-D", "gender": texttospeech.SsmlVoiceGender.MALE},    # Male
    {"name": "fr-FR-Wavenet-E", "gender": texttospeech.SsmlVoiceGender.FEMALE},  # Female
]

# Synthesize each phrase
for idx, text in enumerate(phrases):
    synthesis_input = texttospeech.SynthesisInput(text=text)

    # Randomly pick one of the 3 voices
    voice_choice = random.choice(voices)

    voice = texttospeech.VoiceSelectionParams(
        language_code="fr-FR",
        name=voice_choice["name"],
        ssml_gender=voice_choice["gender"]
    )

    # Audio config
    audio_config = texttospeech.AudioConfig(
        audio_encoding=texttospeech.AudioEncoding.LINEAR16,  # WAV PCM 16kHz
        speaking_rate=1.0
    )

    response = client.synthesize_speech(
        input=synthesis_input, voice=voice, audio_config=audio_config
    )

    # Save to file
    output_path = os.path.join(output_dir, f"phrase_{idx:03d}.wav")
    with open(output_path, "wb") as out:
        out.write(response.audio_content)

    print(f"✅ Saved {output_path} with voice {voice_choice['name']}")


✅ Saved ../data/original_wavs/phrase_000.wav with voice fr-FR-Wavenet-D
✅ Saved ../data/original_wavs/phrase_001.wav with voice fr-FR-Wavenet-E
✅ Saved ../data/original_wavs/phrase_002.wav with voice fr-FR-Wavenet-D
✅ Saved ../data/original_wavs/phrase_003.wav with voice fr-FR-Wavenet-E
✅ Saved ../data/original_wavs/phrase_004.wav with voice fr-FR-Wavenet-B
✅ Saved ../data/original_wavs/phrase_005.wav with voice fr-FR-Wavenet-B
✅ Saved ../data/original_wavs/phrase_006.wav with voice fr-FR-Wavenet-B
✅ Saved ../data/original_wavs/phrase_007.wav with voice fr-FR-Wavenet-E
✅ Saved ../data/original_wavs/phrase_008.wav with voice fr-FR-Wavenet-E
✅ Saved ../data/original_wavs/phrase_009.wav with voice fr-FR-Wavenet-E
✅ Saved ../data/original_wavs/phrase_010.wav with voice fr-FR-Wavenet-B
✅ Saved ../data/original_wavs/phrase_011.wav with voice fr-FR-Wavenet-E
✅ Saved ../data/original_wavs/phrase_012.wav with voice fr-FR-Wavenet-D
✅ Saved ../data/original_wavs/phrase_013.wav with voice fr-FR-Wa

In [ ]:
import os
import json

# Paths
output_metadata_path = "../data/synthetic/transcripts/reference_transcripts.json"
os.makedirs(os.path.dirname(output_metadata_path), exist_ok=True)
wavs_dir = "../data/synthetic/samples/wav"
text_path = "../data/synthetic/synthetic_medical_corpus.txt"

# Load synthetic text
with open(text_path, "r", encoding="utf-8") as f:
    phrases = [line.strip() for line in f.readlines() if line.strip()]

# Get list of generated wav filenames
wav_filenames = sorted([fn for fn in os.listdir(wavs_dir) if fn.endswith(".wav")])

# Sanity check
assert len(wav_filenames) == len(phrases), "Mismatch between number of wav files and phrases!"

# Create mapping
metadata = {wav_fn: phrase for wav_fn, phrase in zip(wav_filenames, phrases)}

# Save as JSON
with open(output_metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"✅ Metadata saved to {output_metadata_path}")

✅ Metadata saved to ../data/metadata.json
